In [1]:
import pandas as pd
import numpy as np

In [2]:
df_mock = pd.read_csv("data/mock/mock_data_join_c03_tscf.csv")

/tmp/ipykernel_93844/1614350122.py:1: DtypeWarning: Columns (13,28,33,53,112,120,128,139,144,160,177,178,179,182,183,184,185,186,187) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mock = pd.read_csv("data/mock/mock_data_join_c03_tscf.csv")


In [5]:
# Load MCC data

mcc_df = pd.read_csv("data/reference/mcc_data.csv", sep=";", dtype={"Code": str})
mcc_df.rename({"Code": "MCC"}, axis=1, inplace=True)

In [9]:
df_mock["MCC"] = df_mock["MCC"].astype("object")

In [10]:
df_mock = df_mock.merge(
    mcc_df[["MCC", "Description", "Transaction Category Code", "MCC Category"]],
    on="MCC",
    how="left",
)

In [11]:
df_mock_selected = df_mock[
    [
        "Transaction Serial No",
        "PANNumber",
        "Transaction Datetime",
        "Transaction Amount",
        "MCC",
        "MCC Category",
        "Country Code",
        "Currency Code",
        "POS Entry Mode",
    ]
].copy()

In [17]:
df_mock_selected[df_mock_selected["PANNumber"].isin([999988325, 115088])].to_csv(
    "data/mock/mock_credit.csv", index=False
)

In [18]:
from src.calculation_features import generate_rolling_features

In [19]:
freq_config = [
    {
        # Transaction count grouped by Card_no/PANNumber
        "type": "frequency",
        "groupby": "PANNumber",
        "amount_col": "Transaction Serial No",
        "groupby_type": "No",
        "groupby_col": None,
        "windows": {
            "900S": "TxnCount_L15M",
            "1H": "TxnCount_L1H",
            "1D": "TxnCount_L1D",
            "7D": "TxnCount_L7D",
            "14D": "TxnCount_L14D",
            "30D": "TxnCount_L30D",
            "90D": "TxnCount_L90D",
        },
    }
]

In [23]:
df_mock_selected["PANNumber"] = df_mock_selected["PANNumber"].astype(str)
df_mock_selected["Transaction Datetime"] = pd.to_datetime(
    df_mock_selected["Transaction Datetime"]
)

# Freq

In [24]:
df_freq = generate_rolling_features(
    df_mock_selected,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=freq_config,
)

Feature Config Progress:   0%|          | 0/1 [00:00<?, ?it/s]/home/gregorius_vidy/gbg_analytics/01 Experimentation/Bespoke ML Pipeline/bespoke-ml/src/calculation_features.py:287: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  .rolling(window, closed="left")
/home/gregorius_vidy/gbg_analytics/01 Experimentation/Bespoke ML Pipeline/bespoke-ml/src/calculation_features.py:287: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .rolling(window, closed="left")
Feature Config Progress: 100%|██████████| 1/1 [00:01<00:00,  1.66s/it]


In [29]:
df_freq[df_freq["PANNumber"].isin(["999988325", "115088"])][
    [
        "Transaction Serial No",
        "PANNumber",
        "Transaction Datetime",
        "TxnCount_L15M",
        "TxnCount_L30D",
    ]
].sort_values(by=["PANNumber", "Transaction Datetime"])

,Transaction Serial No,PANNumber,Transaction Datetime,TxnCount_L15M,TxnCount_L30D
20814,981473574,115088,2024-09-17 08:28:47,0.0,0.0
20822,981473594,115088,2024-09-17 08:28:51,1.0,1.0
20808,981473673,115088,2024-09-17 08:29:12,2.0,2.0
20819,981473692,115088,2024-09-17 08:29:16,3.0,3.0
24086,995085944,115088,2024-10-03 11:07:57,0.0,4.0
12332,1093986107,115088,2025-03-02 11:28:32,0.0,0.0
19948,974909877,999988325,2024-09-08 02:59:42,0.0,0.0
25668,999590208,999988325,2024-10-09 11:51:10,0.0,0.0
25670,999590353,999988325,2024-10-09 11:51:40,1.0,1.0
25671,999590358,999988325,2024-10-09 11:51:40,1.0,1.0


# Unique Count

In [31]:
df_mock_selected["PANNumber Num"], uniques = df_mock_selected["PANNumber"].factorize()

unique_count_config = [
    {
        # Count unique (distinct) Card_no/PANNumber grouped by MCC
        "type": "unique",
        "groupby": "MCC",
        "count_col": "PANNumber Num",
        "windows": {
            "900S": "CntUnique_CardNo_by_MCC_L15M",
            "1H": "CntUnique_CardNo_by_MCC_L1H",
            "1D": "CntUnique_CardNo_by_MCC_L1D",
            "7D": "CntUnique_CardNo_by_MCC_L7D",
            "14D": "CntUnique_CardNo_by_MCC_L14D",
            "30D": "CntUnique_CardNo_by_MCC_L30D",
            "90D": "CntUnique_CardNo_by_MCC_L90D",
        },
    },
]

In [43]:
df_mock_selected["MCC"] = df_mock_selected["MCC"].astype(str)
df_mock_selected["MCC"] = df_mock_selected["MCC"].fillna("-999")

In [44]:
df_unique_count_selected = generate_rolling_features(
    df_mock_selected[df_mock_selected["PANNumber"].isin(["999988325", "115088"])],
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=unique_count_config,
)

Feature Config Progress:   0%|          | 0/1 [00:00<?, ?it/s]/home/gregorius_vidy/gbg_analytics/01 Experimentation/Bespoke ML Pipeline/bespoke-ml/src/calculation_features.py:403: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset[datetime_col] = pd.to_datetime(dataset[datetime_col])
/home/gregorius_vidy/gbg_analytics/01 Experimentation/Bespoke ML Pipeline/bespoke-ml/src/calculation_features.py:414: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  .rolling(window=window, closed="left", min_periods=1)
/home/gregorius_vidy/gbg_analytics/01 Experimentation/Bespoke ML Pipeline/bespoke-ml/src/calculation_features.py:414: FutureWarning: 'S' is deprecated and will be removed in a future version, plea

In [45]:
df_unique_count_selected[
    df_unique_count_selected["PANNumber"].isin(["999988325", "115088"])
][
    [
        "Transaction Serial No",
        "PANNumber",
        "MCC",
        "Transaction Datetime",
        "CntUnique_CardNo_by_MCC_L15M",
        "CntUnique_CardNo_by_MCC_L30D",
    ]
].sort_values(
    by=["PANNumber", "Transaction Datetime"]
)

,Transaction Serial No,PANNumber,MCC,Transaction Datetime,CntUnique_CardNo_by_MCC_L15M,CntUnique_CardNo_by_MCC_L30D
3,981473574,115088,7311.0,2024-09-17 08:28:47,NaN,NaN
5,981473594,115088,7311.0,2024-09-17 08:28:51,1.0,1.0
2,981473673,115088,7311.0,2024-09-17 08:29:12,1.0,1.0
4,981473692,115088,7311.0,2024-09-17 08:29:16,1.0,1.0
6,995085944,115088,7311.0,2024-10-03 11:07:57,NaN,1.0
0,1093986107,115088,7311.0,2025-03-02 11:28:32,NaN,NaN
1,974909877,999988325,4722.0,2024-09-08 02:59:42,NaN,NaN
9,999590208,999988325,-999,2024-10-09 11:51:10,NaN,NaN
10,999590353,999988325,7311.0,2024-10-09 11:51:40,NaN,1.0
11,999590358,999988325,-999,2024-10-09 11:51:40,1.0,1.0
